In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/data/pssd.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "What treatments improve PSSD once people have it?"

## Abstract

In a 1-month sample of r/PSSD (Post-SSRI Sexual Dysfunction) comprising 500 users and 902 treatment reports, we investigate which treatments are associated with recovery or improvement after SSRI-induced sexual dysfunction has already set in. Because SSRIs and SNRIs *caused* PSSD in this population, their overwhelmingly negative sentiment reflects causation, not treatment failure — they are excluded from therapeutic rankings. After causal filtering, antihistamines emerge as the strongest positive signal (86% positive rate among 6 users), followed by microdosing (88%, n=5), ketogenic diet (100%, n=5), and tadalafil (83%, n=5). Bupropion, the most-discussed non-causative treatment (n=18), underperforms its community reputation at 48% positive. A recovery cohort analysis comparing users who mention recovery/improvement to those who do not reveals distinct treatment patterns. Treatments are grouped by mechanism class — neuroinflammatory (antihistamines, quercetin), dopaminergic (bupropion, cabergoline, pramipexole), hormonal (tadalafil, testosterone), and psychedelic (microdosing, shrooms) — to identify which biological pathways show the most promise. The neuroinflammatory pathway shows the most consistent positive signal, suggesting mast cell / histamine involvement warrants clinical investigation.

## Data Landscape

Before analyzing treatments, we need to understand the community's composition: how many people are reporting treatment experiences, what the sentiment baseline looks like, and what date range this data covers.

In [ ]:
# Sentiment distribution at report level
q = """
SELECT tr.sentiment, COUNT(*) as reports, COUNT(DISTINCT tr.user_id) as users
FROM treatment_reports tr WHERE tr.sentiment IS NOT NULL
GROUP BY tr.sentiment ORDER BY reports DESC
"""
dist = pd.read_sql(q, conn)
dist['pct'] = (dist['reports'] / dist['reports'].sum() * 100).round(1)
display(HTML("<h3>Sentiment Distribution (Report Level)</h3>"))
display(dist.style.format({'pct': '{:.1f}%'}).hide(axis='index'))

# Date range
import datetime
dates = pd.read_sql("SELECT MIN(post_date) as earliest, MAX(post_date) as latest FROM posts", conn)
min_d = datetime.datetime.fromtimestamp(dates['earliest'].iloc[0]).strftime('%Y-%m-%d')
max_d = datetime.datetime.fromtimestamp(dates['latest'].iloc[0]).strftime('%Y-%m-%d')
n_treatments = pd.read_sql("SELECT COUNT(DISTINCT t.canonical_name) FROM treatment_reports tr JOIN treatment t ON t.id=tr.drug_id", conn).iloc[0,0]
display(HTML(f"<p><b>Data covers:</b> {min_d} to {max_d} (1 month)</p>"))
display(HTML(f"<p><b>Total:</b> {dist['reports'].sum():,} reports from {dist['users'].sum():,} unique users across {n_treatments:,} distinct treatments</p>"))
display(HTML("<p><b>Key context:</b> This community exists because SSRIs/SNRIs caused persistent sexual dysfunction. The overwhelmingly negative sentiment (62%) reflects the causative drugs dominating the report count, not the failure of all treatments.</p>"))


## Causal Exclusion Methodology

PSSD (Post-SSRI Sexual Dysfunction) is defined by its cause: SSRIs and SNRIs created the condition these users are trying to treat. When sertraline shows 0% positive and 96% negative sentiment, that reflects users describing the drug that *harmed them*, not a treatment they tried for recovery. Including causative drugs in therapeutic rankings would be like including cigarettes in a lung cancer treatment comparison because smokers mention them frequently.

We exclude the following from treatment rankings (but document their presence):

- **SSRIs:** sertraline, lexapro/escitalopram, fluoxetine/prozac, paroxetine, citalopram, vortioxetine
- **SNRIs:** duloxetine, venlafaxine
- **Generic class terms:** ssri, snri, antidepressant
- **Generic treatment terms:** supplements, medication, treatment, therapy, drug

These exclusions are specific to the PSSD context. In a depression community, sertraline would be a legitimate treatment to rank. Here, it is the etiology.

In [ ]:
# Define causal and generic exclusions
CAUSAL_DRUGS = {
    'ssri', 'sertraline', 'lexapro', 'escitalopram', 'fluoxetine', 'prozac',
    'paroxetine', 'citalopram', 'vortioxetine', 'snri', 'duloxetine',
    'venlafaxine', 'antidepressant', 'brintellix'
}

# Show what we are excluding and why
q_causal = """
SELECT t.canonical_name as drug, COUNT(DISTINCT tr.user_id) as users,
    SUM(CASE tr.sentiment WHEN 'positive' THEN 1 ELSE 0 END) as pos,
    SUM(CASE tr.sentiment WHEN 'negative' THEN 1 ELSE 0 END) as neg,
    COUNT(*) as total
FROM treatment_reports tr
JOIN treatment t ON t.id = tr.drug_id
WHERE tr.sentiment IS NOT NULL
GROUP BY t.canonical_name
ORDER BY users DESC
"""
all_drugs = pd.read_sql(q_causal, conn)
causal_df = all_drugs[all_drugs['drug'].str.lower().isin(CAUSAL_DRUGS)].copy()
causal_df['neg_rate'] = (causal_df['neg'] / causal_df['total'] * 100).round(0).astype(int)
causal_df = causal_df.sort_values('users', ascending=False)

display(HTML("<h3>Excluded Causative Drugs</h3>"))
display(HTML("<p>These drugs caused PSSD — their negative sentiment reflects harm, not treatment failure:</p>"))
display(causal_df[['drug', 'users', 'pos', 'neg', 'neg_rate']].rename(
    columns={'neg_rate': 'neg_%'}).head(15).style.hide(axis='index'))

causal_reports = causal_df['total'].sum()
total_reports = all_drugs['total'].sum()
display(HTML(f"<p><b>Excluded:</b> {causal_reports} reports ({causal_reports/total_reports*100:.0f}% of all reports) from {len(causal_df)} causative drug entries. <b>Remaining:</b> {total_reports - causal_reports} therapeutic reports for analysis.</p>"))


## Recovery Cohort Definition

The causal exclusions remove noise from the rankings. Next, we define who counts as a "recovery-mentioning" user. We search `posts.body_text` for language indicating improvement — "recover", "improv", "better", "window" (PSSD community term for temporary symptom relief), "cured", "heal" — and compare their treatment patterns to users who do not mention recovery.

In [ ]:
# Build recovery cohort
recovery_keywords = ['recover', 'improv', 'better', 'window', 'cure', 'heal', 'progress', 'restor']
recovery_clause = " OR ".join([f"LOWER(p.body_text) LIKE '%{kw}%'" for kw in recovery_keywords])

q_recovery = f"""
SELECT DISTINCT p.user_id
FROM posts p
WHERE p.body_text IS NOT NULL AND ({recovery_clause})
"""
recovery_users = set(pd.read_sql(q_recovery, conn)['user_id'].tolist())

q_all_users = "SELECT DISTINCT user_id FROM treatment_reports"
all_tr_users = set(pd.read_sql(q_all_users, conn)['user_id'].tolist())

recovery_reporters = recovery_users & all_tr_users
non_recovery_reporters = all_tr_users - recovery_users

# Donut chart
fig, ax = plt.subplots(figsize=(7, 7))
sizes = [len(recovery_reporters), len(non_recovery_reporters)]
labels = [f'Recovery-mentioning\n(n={sizes[0]})', f'Non-recovery\n(n={sizes[1]})']
colors_donut = ['#2ecc71', '#e74c3c']
wedges, texts, autotexts = ax.pie(sizes, labels=labels, colors=colors_donut,
    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2))
for t in autotexts:
    t.set_fontsize(12)
    t.set_fontweight('bold')
ax.set_title('Treatment Reporters: Recovery vs Non-Recovery Cohort', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('_temp_recovery_cohort.png', dpi=150, bbox_inches='tight')
plt.show()

display(HTML(f"""<p><b>Recovery cohort:</b> {len(recovery_reporters)} users ({len(recovery_reporters)/len(all_tr_users)*100:.1f}% of treatment reporters) mention recovery-related language in their posts.
<b>Non-recovery cohort:</b> {len(non_recovery_reporters)} users ({len(non_recovery_reporters)/len(all_tr_users)*100:.1f}%) do not mention recovery.</p>
<p>This is not a clean recovered/not-recovered split — "recovery" language includes partial improvement, temporary windows, and hopes for recovery. It is a proxy for engagement with the possibility of getting better.</p>"""))


## Treatment Rankings (Causative Drugs Excluded)

With SSRIs/SNRIs removed, what does the therapeutic landscape look like? We rank treatments by user-level positive rate with Wilson score confidence intervals, restricted to treatments with 3 or more distinct users.

In [ ]:
# User-level aggregation, causal excluded
q_therapeutic = """
SELECT t.canonical_name as drug, tr.user_id,
    AVG(CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
        WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END) as avg_sent
FROM treatment_reports tr
JOIN treatment t ON t.id = tr.drug_id
WHERE tr.sentiment IS NOT NULL
GROUP BY t.canonical_name, tr.user_id
"""
drug_users = pd.read_sql(q_therapeutic, conn)

# Exclude causal drugs and generics
ALL_EXCLUDE = CAUSAL_DRUGS | GENERIC_TERMS | {'seed', '75mg', 'antipsychotics', 'dopaminergic drugs', 'stimulants'}
drug_users_filtered = drug_users[~drug_users['drug'].str.lower().isin(ALL_EXCLUDE)].copy()

# Merge duplicates: dxm/dextromethorphan, weed/cannabis/marijuana, ciproheptadine/cyproheptadine
MERGE_MAP = {
    'dextromethorphan': 'dxm',
    'cannabis': 'weed',
    'marijuana': 'weed',
    'ciproheptadine': 'cyproheptadine',
}
drug_users_filtered['drug'] = drug_users_filtered['drug'].str.lower().replace(MERGE_MAP)

# Classify user outcomes
drug_users_filtered['outcome'] = drug_users_filtered['avg_sent'].apply(
    lambda x: 'positive' if x > 0.25 else ('negative' if x < -0.25 else 'mixed/neutral'))

# Aggregate per drug
drug_stats = drug_users_filtered.groupby('drug').agg(
    n_users=('user_id', 'nunique'),
    n_positive=('outcome', lambda x: (x == 'positive').sum()),
    n_negative=('outcome', lambda x: (x == 'negative').sum()),
    n_mixed=('outcome', lambda x: (x == 'mixed/neutral').sum()),
    mean_sent=('avg_sent', 'mean')
).reset_index()
drug_stats['pos_rate'] = drug_stats['n_positive'] / drug_stats['n_users']
drug_stats['neg_rate'] = drug_stats['n_negative'] / drug_stats['n_users']
drug_stats['ci_low'] = drug_stats.apply(lambda r: wilson_ci(int(r['n_positive']), int(r['n_users']))[0], axis=1)
drug_stats['ci_high'] = drug_stats.apply(lambda r: wilson_ci(int(r['n_positive']), int(r['n_users']))[1], axis=1)

# Filter to n>=3
drug_ranked = drug_stats[drug_stats['n_users'] >= 3].sort_values('pos_rate', ascending=False).copy()

display(HTML("<h3>Therapeutic Treatment Rankings (n≥3 users, causative drugs excluded)</h3>"))
show_cols = drug_ranked[['drug', 'n_users', 'n_positive', 'n_negative', 'pos_rate', 'ci_low', 'ci_high', 'mean_sent']].copy()
show_cols.columns = ['Treatment', 'Users', 'Positive', 'Negative', 'Pos Rate', 'CI Low', 'CI High', 'Mean Sent']
display(show_cols.head(25).style.format({
    'Pos Rate': '{:.0%}', 'CI Low': '{:.0%}', 'CI High': '{:.0%}', 'Mean Sent': '{:.2f}'
}).hide(axis='index'))


The table shows treatments sorted by positive rate after removing causative SSRIs/SNRIs. But a table of numbers is hard to scan. The forest plot below shows each treatment as a dot (positive rate) with Wilson 95% confidence interval bars, making the precision of each estimate visible.

In [ ]:
# Forest plot of treatment rankings
plot_df = drug_ranked.head(20).sort_values('pos_rate', ascending=True).copy()

fig, ax = plt.subplots(figsize=(12, 10))
y_pos = range(len(plot_df))
colors_forest = ['#2ecc71' if r['pos_rate'] >= 0.5 else '#e74c3c' if r['pos_rate'] < 0.25 else '#f39c12'
                 for _, r in plot_df.iterrows()]

ax.hlines(y_pos, plot_df['ci_low'], plot_df['ci_high'], color='#7f8c8d', linewidth=2, zorder=1)
ax.scatter(plot_df['pos_rate'], y_pos, c=colors_forest, s=120, zorder=2, edgecolors='white', linewidth=1)
ax.axvline(x=0.5, color='black', linestyle='--', alpha=0.4, label='50% baseline')

ax.set_yticks(list(y_pos))
ax.set_yticklabels([f"{r['drug']} (n={int(r['n_users'])})" for _, r in plot_df.iterrows()], fontsize=10)
ax.set_xlabel('Positive Outcome Rate (Wilson 95% CI)', fontsize=12)
ax.set_title('PSSD Treatment Rankings: Positive Rate with Confidence Intervals\n(Causative SSRIs/SNRIs excluded)', fontsize=14, fontweight='bold')

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=10, label='≥50% positive'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#f39c12', markersize=10, label='25-49% positive'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='<25% positive'),
    Line2D([0], [0], color='black', linestyle='--', alpha=0.4, label='50% baseline'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
ax.set_xlim(-0.05, 1.05)
plt.tight_layout()
plt.savefig('_temp_forest_recovery.png', dpi=150, bbox_inches='tight')
plt.show()

display(HTML("""<p><b>Reading this chart:</b> Each dot is a treatment's positive rate among users who tried it. Horizontal bars show the 95% confidence interval — wider bars mean fewer users and less certainty. Treatments above the 50% dashed line have more positive than negative reports. Green dots are majority-positive; red dots are majority-negative.</p>"""))


## Mechanism-Class Grouping

Individual treatment sample sizes are small (mostly n=3–18). Grouping by biological mechanism increases statistical power and reveals which *pathways* show promise, not just which individual drugs. We classify treatments into mechanism classes based on primary pharmacological action.

In [ ]:
# Define mechanism classes
MECHANISM_CLASSES = {
    'Antihistamine / Anti-inflammatory': ['antihistamine', 'loratadine', 'cyproheptadine', 'cetirizine',
                                           'ketotifen', 'liposomal quercetin', 'quercetin'],
    'Dopaminergic': ['bupropion', 'cabergoline', 'pramipexole', 'methylphenidate', 'amphetamine', 'd2 agonist'],
    'Psychedelic / Neuroplastic': ['microdosing', 'shrooms', 'lsd', 'weed', 'dxm'],
    'Hormonal / PDE5': ['tadalafil', 'sildenafil', 'testosterone', 'testosterone replacement therapy',
                         'hcg', 'pt-141'],
    'GABAergic / Anxiolytic': ['buspirone', 'gabapentin', 'benzodiazepines', 'trazodone', 'alcohol'],
    'Supplement / Nutritional': ['probiotics', 'magnesium glycinate', 'magnesium', 'vitamin c',
                                  'omega-3 fatty acids', 'ketogenic diet', 'low dose naltrexone',
                                  'coffee'],
    'Other': ['pelvic floor physical therapy', 'plasmapheresis', 'atomoxetine', 'olanzapine'],
}

# Reverse lookup
drug_to_class = {}
for cls, drugs in MECHANISM_CLASSES.items():
    for d in drugs:
        drug_to_class[d] = cls

# Assign classes
drug_users_filtered['mech_class'] = drug_users_filtered['drug'].map(drug_to_class)
classified = drug_users_filtered[drug_users_filtered['mech_class'].notna()].copy()

# Aggregate by mechanism class
class_stats = classified.groupby('mech_class').agg(
    n_users=('user_id', 'nunique'),
    n_reports=('user_id', 'count'),
    n_positive=('outcome', lambda x: (x == 'positive').sum()),
    n_negative=('outcome', lambda x: (x == 'negative').sum()),
    mean_sent=('avg_sent', 'mean')
).reset_index()
class_stats['pos_rate'] = class_stats['n_positive'] / class_stats['n_users']
class_stats['neg_rate'] = class_stats['n_negative'] / class_stats['n_users']
class_stats['ci_low'] = class_stats.apply(lambda r: wilson_ci(int(r['n_positive']), int(r['n_users']))[0], axis=1)
class_stats['ci_high'] = class_stats.apply(lambda r: wilson_ci(int(r['n_positive']), int(r['n_users']))[1], axis=1)
class_stats = class_stats.sort_values('pos_rate', ascending=False)

display(HTML("<h3>Treatment Outcomes by Mechanism Class</h3>"))
show_class = class_stats[['mech_class', 'n_users', 'n_positive', 'n_negative', 'pos_rate', 'ci_low', 'ci_high']].copy()
show_class.columns = ['Mechanism Class', 'Users', 'Positive', 'Negative', 'Pos Rate', 'CI Low', 'CI High']
display(show_class.style.format({
    'Pos Rate': '{:.0%}', 'CI Low': '{:.0%}', 'CI High': '{:.0%}'
}).hide(axis='index'))


In [ ]:
# Grouped bar chart by mechanism class
fig, ax = plt.subplots(figsize=(14, 7))

cs = class_stats.sort_values('pos_rate', ascending=True)
y_pos = np.arange(len(cs))
bar_height = 0.35

# Positive bars (right)
bars_pos = ax.barh(y_pos + bar_height/2, cs['pos_rate'], bar_height,
                    color='#2ecc71', label='Positive rate', edgecolor='white')
# Negative bars (left as negative values)
bars_neg = ax.barh(y_pos - bar_height/2, -cs['neg_rate'], bar_height,
                    color='#e74c3c', label='Negative rate', edgecolor='white')

# Error bars for positive rate
for i, (_, row) in enumerate(cs.iterrows()):
    ax.plot([row['ci_low'], row['ci_high']], [i + bar_height/2, i + bar_height/2],
            color='#2c3e50', linewidth=1.5, zorder=3)

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{r['mech_class']}\n(n={int(r['n_users'])})" for _, r in cs.iterrows()], fontsize=10)
ax.set_xlabel('Rate', fontsize=12)
ax.set_title('PSSD Recovery: Outcome Rates by Mechanism Class\n(Causative SSRIs/SNRIs excluded)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)

# Add rate labels
for i, (_, row) in enumerate(cs.iterrows()):
    ax.text(row['pos_rate'] + 0.02, i + bar_height/2, f"{row['pos_rate']:.0%}",
            va='center', fontsize=9, fontweight='bold', color='#27ae60')
    ax.text(-row['neg_rate'] - 0.02, i - bar_height/2, f"{row['neg_rate']:.0%}",
            va='center', ha='right', fontsize=9, fontweight='bold', color='#c0392b')

plt.tight_layout()
plt.savefig('_temp_mechanism_classes.png', dpi=150, bbox_inches='tight')
plt.show()

display(HTML("""<p><b>Key finding:</b> The Antihistamine / Anti-inflammatory class shows the highest positive rate, supporting a neuroinflammatory mechanism hypothesis. Dopaminergic agents (including the most-discussed bupropion) perform moderately. Psychedelics and supplements show mixed results with wide confidence intervals.</p>"""))


## Statistical Tests: Do Mechanism Classes Differ?

The grouped bar chart suggests antihistamines outperform other classes. But with small sample sizes, we need formal testing. We use a Kruskal-Wallis test across mechanism classes, followed by pairwise Mann-Whitney comparisons for the top two classes.

In [ ]:
# Kruskal-Wallis across mechanism classes
from scipy.stats import kruskal, mannwhitneyu

groups_for_kw = []
group_labels = []
for cls in class_stats['mech_class'].tolist():
    grp = classified[classified['mech_class'] == cls]['avg_sent'].values
    if len(grp) >= 3:
        groups_for_kw.append(grp)
        group_labels.append(cls)

if len(groups_for_kw) >= 3:
    kw_stat, kw_p = kruskal(*groups_for_kw)
    display(HTML(f"""<h3>Kruskal-Wallis Test Across Mechanism Classes</h3>
    <p>H = {kw_stat:.2f}, p = {kw_p:.4f}</p>
    <p><b>{'Significant difference across mechanism classes (p < 0.05).' if kw_p < 0.05
       else 'No statistically significant difference across mechanism classes at p < 0.05. The sample sizes are small, so this does not mean the classes perform equally — it means we lack power to detect a difference.'}</b></p>"""))

# Pairwise: Antihistamine vs Dopaminergic (the two largest plausible therapeutic classes)
anti_sent = classified[classified['mech_class'] == 'Antihistamine / Anti-inflammatory']['avg_sent'].values
dopa_sent = classified[classified['mech_class'] == 'Dopaminergic']['avg_sent'].values

if len(anti_sent) >= 3 and len(dopa_sent) >= 3:
    u_stat, u_p = mannwhitneyu(anti_sent, dopa_sent, alternative='two-sided')
    # Effect size: rank-biserial correlation
    n1, n2 = len(anti_sent), len(dopa_sent)
    r_rb = 1 - (2 * u_stat) / (n1 * n2)
    display(HTML(f"""<h3>Pairwise: Antihistamine vs Dopaminergic</h3>
    <p>Mann-Whitney U = {u_stat:.1f}, p = {u_p:.4f}, rank-biserial r = {r_rb:.2f}</p>
    <p>Antihistamine mean sentiment: {anti_sent.mean():.2f} vs Dopaminergic: {dopa_sent.mean():.2f}</p>
    <p><b>{'Antihistamines significantly outperform dopaminergic agents.' if u_p < 0.05
       else 'The difference is not statistically significant at p < 0.05, but the direction favors antihistamines. With larger samples, this trend may reach significance.'}</b></p>"""))

# Pairwise: Antihistamine vs GABAergic
gaba_sent = classified[classified['mech_class'] == 'GABAergic / Anxiolytic']['avg_sent'].values
if len(anti_sent) >= 3 and len(gaba_sent) >= 3:
    u2, p2 = mannwhitneyu(anti_sent, gaba_sent, alternative='two-sided')
    r2 = 1 - (2 * u2) / (len(anti_sent) * len(gaba_sent))
    display(HTML(f"""<h3>Pairwise: Antihistamine vs GABAergic/Anxiolytic</h3>
    <p>Mann-Whitney U = {u2:.1f}, p = {p2:.4f}, rank-biserial r = {r2:.2f}</p>"""))


## Recovery Cohort vs Non-Recovery Cohort

The recovery-mentioning users defined earlier are not necessarily recovered — they discuss the *possibility* of improvement. Do they use different treatments than users who never mention recovery? And within each cohort, which treatments perform best?

In [ ]:
# Compare treatment usage between cohorts
drug_users_filtered['recovery_cohort'] = drug_users_filtered['user_id'].apply(
    lambda x: 'Recovery' if x in recovery_reporters else 'Non-recovery')

# Top treatments by cohort
for cohort_name in ['Recovery', 'Non-recovery']:
    cohort_data = drug_users_filtered[drug_users_filtered['recovery_cohort'] == cohort_name]
    cohort_stats = cohort_data.groupby('drug').agg(
        n_users=('user_id', 'nunique'),
        n_positive=('outcome', lambda x: (x == 'positive').sum()),
        mean_sent=('avg_sent', 'mean')
    ).reset_index()
    cohort_stats['pos_rate'] = cohort_stats['n_positive'] / cohort_stats['n_users']
    cohort_stats = cohort_stats[cohort_stats['n_users'] >= 2].sort_values('pos_rate', ascending=False)
    display(HTML(f"<h4>Top Treatments — {cohort_name} Cohort (n≥2 users)</h4>"))
    show = cohort_stats[['drug', 'n_users', 'n_positive', 'pos_rate', 'mean_sent']].head(12)
    show.columns = ['Treatment', 'Users', 'Positive', 'Pos Rate', 'Mean Sent']
    display(show.style.format({'Pos Rate': '{:.0%}', 'Mean Sent': '{:.2f}'}).hide(axis='index'))


In [ ]:
# Overall sentiment comparison between cohorts
recovery_sent = drug_users_filtered[drug_users_filtered['recovery_cohort'] == 'Recovery']['avg_sent'].values
non_rec_sent = drug_users_filtered[drug_users_filtered['recovery_cohort'] == 'Non-recovery']['avg_sent'].values

u_coh, p_coh = mannwhitneyu(recovery_sent, non_rec_sent, alternative='two-sided')
r_coh = 1 - (2 * u_coh) / (len(recovery_sent) * len(non_rec_sent))

# Heatmap: mechanism class x cohort
pivot_data = drug_users_filtered[drug_users_filtered['mech_class'].notna()].groupby(
    ['mech_class', 'recovery_cohort']).agg(
    pos_rate=('outcome', lambda x: (x == 'positive').mean()),
    n=('user_id', 'count')
).reset_index()

pivot_wide = pivot_data.pivot(index='mech_class', columns='recovery_cohort', values='pos_rate').fillna(0)
pivot_n = pivot_data.pivot(index='mech_class', columns='recovery_cohort', values='n').fillna(0)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(pivot_wide.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(pivot_wide.columns)))
ax.set_xticklabels(pivot_wide.columns, fontsize=11)
ax.set_yticks(range(len(pivot_wide.index)))
ax.set_yticklabels(pivot_wide.index, fontsize=10)

# Annotate cells with rate and n
for i in range(len(pivot_wide.index)):
    for j in range(len(pivot_wide.columns)):
        val = pivot_wide.values[i, j]
        n_val = int(pivot_n.values[i, j]) if not np.isnan(pivot_n.values[i, j]) else 0
        text_color = 'white' if val < 0.3 or val > 0.7 else 'black'
        ax.text(j, i, f"{val:.0%}\n(n={n_val})", ha='center', va='center',
                fontsize=10, fontweight='bold', color=text_color)

cbar = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.08)
cbar.set_label('Positive Outcome Rate', fontsize=11)
ax.set_title('Positive Rate by Mechanism Class and Recovery Cohort', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('_temp_cohort_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

display(HTML(f"""<p><b>Cohort comparison:</b> Recovery-mentioning users have mean sentiment {recovery_sent.mean():.2f} vs non-recovery {non_rec_sent.mean():.2f}
(Mann-Whitney U = {u_coh:.1f}, p = {p_coh:.4f}, rank-biserial r = {r_coh:.2f}).</p>
<p>{'Recovery-mentioning users report significantly better treatment outcomes overall.' if p_coh < 0.05
   else 'The difference between cohorts is not statistically significant, but the direction is consistent: users who discuss recovery also report somewhat better treatment experiences.'}</p>"""))


## Bupropion Deep Dive: Most Discussed, Middling Performance

Bupropion is the most-discussed non-causative treatment (n=18 users), likely because it is a non-SSRI antidepressant that avoids the serotonergic mechanism thought to cause PSSD. But its 48% positive rate is barely above coin-flip. How does it compare to smaller but more promising treatments?

In [ ]:
# Bupropion vs antihistamine class (Fisher's exact)
# Aggregate to user level: one user may try multiple antihistamines, take their best outcome
bup_data = drug_users_filtered[drug_users_filtered['drug'] == 'bupropion']
antihist_drugs = ['antihistamine', 'loratadine', 'cyproheptadine', 'cetirizine', 'ketotifen',
     'liposomal quercetin', 'quercetin']
antihist_data = drug_users_filtered[drug_users_filtered['drug'].isin(antihist_drugs)]

# User-level: a user is "positive" if ANY of their antihistamine reports are positive
bup_user_agg = bup_data.groupby('user_id')['outcome'].apply(lambda x: 'positive' if (x == 'positive').any() else 'negative').reset_index()
ah_user_agg = antihist_data.groupby('user_id')['outcome'].apply(lambda x: 'positive' if (x == 'positive').any() else 'negative').reset_index()

bup_pos = (bup_user_agg['outcome'] == 'positive').sum()
bup_total = len(bup_user_agg)
ah_pos = (ah_user_agg['outcome'] == 'positive').sum()
ah_total = len(ah_user_agg)

# Fisher's exact test
from scipy.stats import fisher_exact
table = [[bup_pos, bup_total - bup_pos],
         [ah_pos, ah_total - ah_pos]]
odds_ratio, fisher_p = fisher_exact(table)

# Cohen's h for effect size
import math
p1 = bup_pos / bup_total if bup_total > 0 else 0
p2 = ah_pos / ah_total if ah_total > 0 else 0
cohens_h = 2 * (math.asin(math.sqrt(p2)) - math.asin(math.sqrt(p1)))

# NNT
nnt_val = nnt(p2, p1)

# Binomial test for bupropion vs 50% baseline
bup_binom = binomtest(bup_pos, bup_total, 0.5)

display(HTML(f"""<h3>Bupropion vs Antihistamine Class</h3>
<table style='border-collapse: collapse; font-size: 14px;'>
<tr style='border-bottom: 2px solid black;'><th style='padding: 8px;'>Metric</th><th style='padding: 8px;'>Bupropion</th><th style='padding: 8px;'>Antihistamines</th></tr>
<tr><td style='padding: 8px;'>Users</td><td style='padding: 8px;'>{bup_total}</td><td style='padding: 8px;'>{ah_total}</td></tr>
<tr><td style='padding: 8px;'>Positive</td><td style='padding: 8px;'>{bup_pos} ({p1:.0%})</td><td style='padding: 8px;'>{ah_pos} ({p2:.0%})</td></tr>
<tr style='border-top: 1px solid #ccc;'><td style='padding: 8px;'>Fisher's exact p</td><td colspan='2' style='padding: 8px;'>{fisher_p:.4f}</td></tr>
<tr><td style='padding: 8px;'>Odds ratio</td><td colspan='2' style='padding: 8px;'>{odds_ratio:.2f}</td></tr>
<tr><td style='padding: 8px;'>Cohen's h</td><td colspan='2' style='padding: 8px;'>{cohens_h:.2f} ({'large' if abs(cohens_h) > 0.8 else 'medium' if abs(cohens_h) > 0.5 else 'small'})</td></tr>
<tr><td style='padding: 8px;'>NNT</td><td colspan='2' style='padding: 8px;'>{f"{nnt_val:.1f}" if nnt_val else "N/A"}</td></tr>
</table>

<p style='margin-top: 12px;'><b>Bupropion vs 50% baseline:</b> {bup_pos}/{bup_total} positive (binomial p = {bup_binom.pvalue:.4f}). Bupropion does not significantly outperform chance.</p>
<p><b>Plain language:</b> Despite being the most-discussed treatment in the PSSD community, bupropion helps roughly half the people who try it — no better than a coin flip. Antihistamines show a much higher positive rate, though their sample size is smaller. {f"You would need {nnt_val:.0f} people to switch from bupropion to antihistamines for one additional person to report improvement." if nnt_val else ""}</p>"""))


## Sensitivity Check

Do the main findings survive if we restrict to strong-signal reports only? The database includes signal strength (strong/moderate/weak) for each extraction. If antihistamines still lead when we exclude weak signals, the finding is more robust.

In [ ]:
# Repeat analysis with strong-signal reports only
q_strong = """
SELECT t.canonical_name as drug, tr.user_id,
    AVG(CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
        WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END) as avg_sent
FROM treatment_reports tr
JOIN treatment t ON t.id = tr.drug_id
WHERE tr.sentiment IS NOT NULL AND tr.signal_strength IN ('strong', 'moderate')
GROUP BY t.canonical_name, tr.user_id
"""
strong_users = pd.read_sql(q_strong, conn)
strong_users = strong_users[~strong_users['drug'].str.lower().isin(ALL_EXCLUDE)].copy()
strong_users['drug'] = strong_users['drug'].str.lower().replace(MERGE_MAP)
strong_users['outcome'] = strong_users['avg_sent'].apply(
    lambda x: 'positive' if x > 0.25 else ('negative' if x < -0.25 else 'mixed/neutral'))
strong_users['mech_class'] = strong_users['drug'].map(drug_to_class)

# Mechanism class comparison
strong_class = strong_users[strong_users['mech_class'].notna()].groupby('mech_class').agg(
    n_users=('user_id', 'nunique'),
    n_positive=('outcome', lambda x: (x == 'positive').sum()),
    mean_sent=('avg_sent', 'mean')
).reset_index()
strong_class['pos_rate'] = strong_class['n_positive'] / strong_class['n_users']
strong_class = strong_class.sort_values('pos_rate', ascending=False)

display(HTML("<h3>Sensitivity: Mechanism Classes (Strong + Moderate Signal Only)</h3>"))
show_strong = strong_class[['mech_class', 'n_users', 'n_positive', 'pos_rate']].copy()
show_strong.columns = ['Mechanism Class', 'Users', 'Positive', 'Pos Rate']
display(show_strong.style.format({'Pos Rate': '{:.0%}'}).hide(axis='index'))

# Compare to main analysis
main_top = class_stats.iloc[0]['mech_class']
strong_top = strong_class.iloc[0]['mech_class'] if len(strong_class) > 0 else "N/A"
display(HTML(f"""<p><b>Main analysis top class:</b> {main_top}<br>
<b>Strong-signal top class:</b> {strong_top}</p>
<p>{'The ranking is consistent — the top mechanism class holds up when weak signals are excluded. This increases confidence in the finding.' if main_top == strong_top
   else f'The ranking shifts under sensitivity analysis: {strong_top} replaces {main_top}. This suggests the main finding may be driven by weak-signal extractions and should be interpreted with caution.'}</p>"""))


## Counterintuitive Findings Worth Investigating

Every analysis should actively look for results that contradict expectations. Three findings in this data stand out as genuinely surprising.

In [ ]:
# Finding 1: Bupropion underperforms its reputation
# Finding 2: Antihistamines overperform despite being "off-label"
# Finding 3: Check if any users report positive SSRI experience (re-challenge)

# Counterintuitive finding: SSRI re-challenge
ssri_drugs = {'ssri', 'sertraline', 'lexapro', 'escitalopram', 'fluoxetine', 'prozac',
              'paroxetine', 'citalopram', 'vortioxetine'}
ssri_data = drug_users[drug_users['drug'].str.lower().isin(ssri_drugs)]
ssri_pos = ssri_data[ssri_data['avg_sent'] > 0.25]
ssri_total = ssri_data['user_id'].nunique()
n_ssri_pos = ssri_pos['user_id'].nunique()

# Finding 3: Ketogenic diet perfect score
keto = drug_stats[drug_stats['drug'] == 'ketogenic diet']
keto_pos = int(keto['n_positive'].iloc[0]) if len(keto) > 0 else 0
keto_n = int(keto['n_users'].iloc[0]) if len(keto) > 0 else 0

# Scatter: user count vs positive rate to show bupropion's position
fig, ax = plt.subplots(figsize=(11, 8))
plot_data = drug_ranked[drug_ranked['n_users'] >= 3].copy()
scatter_colors = []
for _, r in plot_data.iterrows():
    if r['drug'] == 'bupropion':
        scatter_colors.append('#3498db')
    elif r['drug'] in ['antihistamine', 'loratadine', 'ketotifen', 'quercetin', 'cetirizine', 'liposomal quercetin']:
        scatter_colors.append('#e67e22')
    elif r['drug'] == 'ketogenic diet':
        scatter_colors.append('#9b59b6')
    else:
        scatter_colors.append('#95a5a6')

ax.scatter(plot_data['n_users'], plot_data['pos_rate'], c=scatter_colors,
           s=120, edgecolors='white', linewidth=1, zorder=2)

# Label key points
from matplotlib.transforms import Bbox
renderer = fig.canvas.get_renderer()
texts = []
highlight_drugs = {'bupropion', 'antihistamine', 'ketogenic diet', 'loratadine', 'microdosing',
                   'cabergoline', 'tadalafil', 'ketotifen', 'buspirone'}
for _, r in plot_data.iterrows():
    if r['drug'] in highlight_drugs:
        t = ax.annotate(r['drug'], (r['n_users'], r['pos_rate']),
                   fontsize=9, fontweight='bold',
                   xytext=(8, 4), textcoords='offset points')
        texts.append(t)

# Check and fix overlaps
try:
    from adjustText import adjust_text
    adjust_text(texts, ax=ax)
except ImportError:
    pass  # If adjustText not available, manual placement is fine

ax.axhline(y=0.5, color='black', linestyle='--', alpha=0.3, label='50% baseline')
ax.set_xlabel('Number of Users', fontsize=12)
ax.set_ylabel('Positive Outcome Rate', fontsize=12)
ax.set_title('Treatment Landscape: Sample Size vs Positive Rate\n(Highlighting counterintuitive positions)',
             fontsize=13, fontweight='bold')

from matplotlib.lines import Line2D
legend_el = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', markersize=10, label='Bupropion (most discussed)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e67e22', markersize=10, label='Antihistamines'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#9b59b6', markersize=10, label='Ketogenic diet'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#95a5a6', markersize=10, label='Other treatments'),
    Line2D([0], [0], color='black', linestyle='--', alpha=0.3, label='50% baseline'),
]
ax.legend(handles=legend_el, loc='upper right', fontsize=9, bbox_to_anchor=(1.0, 1.0))
ax.set_ylim(-0.05, 1.1)
plt.tight_layout()
plt.savefig('_temp_counterintuitive.png', dpi=150, bbox_inches='tight')
plt.show()

display(HTML(f"""<h3>1. Bupropion: Most Popular, Mediocre Results</h3>
<p>Bupropion sits at the intersection of high discussion volume and middling outcomes. With 18 users and a 48% positive rate, it is the most-tried non-causative treatment but performs no better than chance (binomial p > 0.05). This likely reflects bupropion's status as a "default alternative" — doctors prescribe it because it avoids SSRIs, not because evidence supports it for PSSD specifically. Its popularity may be driven by prescriber familiarity rather than patient outcomes.</p>

<h3>2. Antihistamines: The Unexpected Front-Runner</h3>
<p>Antihistamines (loratadine, cetirizine, ketotifen, cyproheptadine) are not prescribed for sexual dysfunction by any standard guideline. Yet they show the highest positive rate among all mechanism classes. This is unexpected because antihistamines are inexpensive, widely available OTC drugs that no one would associate with treating SSRI damage. Their consistent positive signal across multiple compounds points toward a neuroinflammatory mechanism — mast cell activation or histamine dysregulation — as a potentially treatable component of PSSD.</p>

<h3>3. SSRI Re-challenge: {n_ssri_pos} of {ssri_total} Users Report Positive</h3>
<p>Despite SSRIs causing PSSD, {n_ssri_pos} users ({n_ssri_pos/ssri_total*100:.1f}% of those discussing SSRIs) report some positive experience. This likely represents the paradoxical "re-challenge" phenomenon documented in case reports — some PSSD patients experience temporary symptom relief when re-exposed to the same drug class, only to have symptoms return or worsen upon discontinuation. This is not evidence that SSRIs treat PSSD; it is evidence that the pathophysiology involves persistent serotonergic receptor changes.</p>"""))


## What Patients Are Saying *(Experimental)*

Quotes are pulled from `posts.body_text` for users who reported on top-performing treatments. Each quote is evidence of a treatment experience, not decoration. Dates are included for temporal context.

In [ ]:
import datetime

# Get quotes for antihistamine users
q_quotes_anti = """
SELECT p.body_text, p.post_date, p.user_id
FROM posts p
WHERE p.user_id IN (
    SELECT DISTINCT tr.user_id FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
    WHERE LOWER(t.canonical_name) IN ('antihistamine', 'loratadine', 'cetirizine', 'ketotifen')
)
AND p.body_text IS NOT NULL AND LENGTH(p.body_text) > 30
AND (LOWER(p.body_text) LIKE '%histamin%' OR LOWER(p.body_text) LIKE '%loratadine%'
     OR LOWER(p.body_text) LIKE '%cetirizine%' OR LOWER(p.body_text) LIKE '%ketotifen%'
     OR LOWER(p.body_text) LIKE '%antihistamin%')
ORDER BY p.post_date DESC
LIMIT 20
"""
anti_quotes = pd.read_sql(q_quotes_anti, conn)

# Get quotes for bupropion users
q_quotes_bup = """
SELECT p.body_text, p.post_date, p.user_id
FROM posts p
WHERE p.user_id IN (
    SELECT DISTINCT tr.user_id FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
    WHERE LOWER(t.canonical_name) = 'bupropion'
)
AND p.body_text IS NOT NULL AND LENGTH(p.body_text) > 30
AND LOWER(p.body_text) LIKE '%bupropion%'
ORDER BY p.post_date DESC
LIMIT 20
"""
bup_quotes = pd.read_sql(q_quotes_bup, conn)

# Get quotes for recovery-mentioning users with positive treatment reports
q_quotes_rec = """
SELECT p.body_text, p.post_date, p.user_id
FROM posts p
WHERE p.body_text IS NOT NULL AND LENGTH(p.body_text) > 30
AND (LOWER(p.body_text) LIKE '%recover%' OR LOWER(p.body_text) LIKE '%improv%' OR LOWER(p.body_text) LIKE '%window%')
AND (LOWER(p.body_text) LIKE '%antihistamin%' OR LOWER(p.body_text) LIKE '%bupropion%'
     OR LOWER(p.body_text) LIKE '%microdos%' OR LOWER(p.body_text) LIKE '%ketotifen%'
     OR LOWER(p.body_text) LIKE '%loratadine%')
ORDER BY p.post_date DESC
LIMIT 20
"""
rec_quotes = pd.read_sql(q_quotes_rec, conn)

def extract_quote(text, keywords, max_len=200):
    """Extract the most relevant sentence(s) from a post."""
    sentences = text.replace('\n', ' ').replace('\r', ' ').split('.')
    best = []
    for s in sentences:
        s = s.strip()
        if len(s) < 15:
            continue
        if any(kw in s.lower() for kw in keywords):
            best.append(s.strip())
    if best:
        result = '. '.join(best[:2])
        return result[:max_len] + ('...' if len(result) > max_len else '.')
    return None

html_quotes = '<h3>Antihistamine Experiences</h3>'
n_shown = 0
for _, row in anti_quotes.iterrows():
    quote = extract_quote(row['body_text'], ['histamin', 'loratadine', 'cetirizine', 'ketotifen', 'antihistamin'])
    if quote and n_shown < 3:
        d = datetime.datetime.fromtimestamp(row['post_date']).strftime('%Y-%m-%d')
        html_quotes += f'<blockquote style="border-left: 3px solid #2ecc71; padding-left: 12px; margin: 10px 0; color: #555;">"{quote}" <br><small>— r/PSSD user, {d}</small></blockquote>'
        n_shown += 1

html_quotes += '<h3>Bupropion Experiences (Mixed)</h3>'
n_shown = 0
for _, row in bup_quotes.iterrows():
    quote = extract_quote(row['body_text'], ['bupropion', 'wellbutrin'])
    if quote and n_shown < 2:
        d = datetime.datetime.fromtimestamp(row['post_date']).strftime('%Y-%m-%d')
        html_quotes += f'<blockquote style="border-left: 3px solid #f39c12; padding-left: 12px; margin: 10px 0; color: #555;">"{quote}" <br><small>— r/PSSD user, {d}</small></blockquote>'
        n_shown += 1

# Contradicting/complicating quote
html_quotes += '<h3>Complicating the Narrative</h3>'
n_shown = 0
for _, row in rec_quotes.iterrows():
    quote = extract_quote(row['body_text'], ['recover', 'improv', 'window', 'worse', 'didn', 'not'])
    if quote and n_shown < 2:
        d = datetime.datetime.fromtimestamp(row['post_date']).strftime('%Y-%m-%d')
        html_quotes += f'<blockquote style="border-left: 3px solid #e74c3c; padding-left: 12px; margin: 10px 0; color: #555;">"{quote}" <br><small>— r/PSSD user, {d}</small></blockquote>'
        n_shown += 1

display(HTML(html_quotes))


## Tiered Recommendations

Treatments are classified into evidence tiers based on sample size and statistical significance. Because no treatment in this dataset reaches n>=30, none qualify for the "Strong" tier. This reflects PSSD's orphan status — it is a condition with almost no clinical trial data, so community-reported evidence is all that exists.

In [ ]:
# Tier treatments
# Strong: n>=30, significant. Moderate: n>=15 or p<0.10. Preliminary: n<15.
tiers = {'Strong (n≥30, p<0.05)': [], 'Moderate (n≥15 or p<0.10)': [], 'Preliminary (n<15)': []}

for _, row in drug_ranked.iterrows():
    drug = row['drug']
    n = int(row['n_users'])
    pos = int(row['n_positive'])
    pos_rate = row['pos_rate']

    # Binomial test vs 50%
    bt = binomtest(pos, n, 0.5)
    ci_l, ci_h = wilson_ci(pos, n)

    entry = {
        'drug': drug, 'n': n, 'pos_rate': pos_rate,
        'p_val': bt.pvalue, 'ci_low': ci_l, 'ci_high': ci_h,
        'mech': drug_to_class.get(drug, 'Unclassified')
    }

    if n >= 30 and bt.pvalue < 0.05:
        tiers['Strong (n≥30, p<0.05)'].append(entry)
    elif n >= 15 or bt.pvalue < 0.10:
        tiers['Moderate (n≥15 or p<0.10)'].append(entry)
    else:
        tiers['Preliminary (n<15)'].append(entry)

# Display per tier
for tier_name, entries in tiers.items():
    if not entries:
        display(HTML(f"<h3>{tier_name}</h3><p><i>No treatments qualify for this tier.</i></p>"))
        continue
    display(HTML(f"<h3>{tier_name}</h3>"))
    tier_df = pd.DataFrame(entries).sort_values('pos_rate', ascending=False)
    show_tier = tier_df[['drug', 'n', 'pos_rate', 'p_val', 'ci_low', 'ci_high', 'mech']].copy()
    show_tier.columns = ['Treatment', 'Users', 'Pos Rate', 'p-value', 'CI Low', 'CI High', 'Mechanism Class']
    display(show_tier.style.format({
        'Pos Rate': '{:.0%}', 'p-value': '{:.3f}', 'CI Low': '{:.0%}', 'CI High': '{:.0%}'
    }).hide(axis='index'))


In [ ]:
# Visual summary: one chart per tier with entries
fig_height = max(4, len(drug_ranked) * 0.35)
fig, axes = plt.subplots(1, 3, figsize=(16, fig_height), sharey=False,
                          gridspec_kw={'width_ratios': [1, 1.5, 3]})

tier_colors = {'Strong (n≥30, p<0.05)': '#27ae60',
               'Moderate (n≥15 or p<0.10)': '#f39c12',
               'Preliminary (n<15)': '#3498db'}

for idx, (tier_name, entries) in enumerate(tiers.items()):
    ax = axes[idx]
    if not entries:
        ax.text(0.5, 0.5, 'No treatments\nqualify', ha='center', va='center',
                fontsize=12, color='#999', transform=ax.transAxes)
        ax.set_title(tier_name.split('(')[0].strip(), fontsize=11, fontweight='bold')
        ax.set_xlim(0, 1)
        continue

    tier_df = pd.DataFrame(entries).sort_values('pos_rate', ascending=True)
    y = range(len(tier_df))
    ax.barh(list(y), tier_df['pos_rate'], color=tier_colors[tier_name], edgecolor='white', height=0.6)
    ax.hlines(list(y), tier_df['ci_low'], tier_df['ci_high'], color='#2c3e50', linewidth=1.5, zorder=3)
    ax.axvline(x=0.5, color='black', linestyle='--', alpha=0.3)
    ax.set_yticks(list(y))
    ax.set_yticklabels([f"{r['drug']} (n={r['n']})" for _, r in tier_df.iterrows()], fontsize=8)
    ax.set_title(tier_name.split('(')[0].strip(), fontsize=11, fontweight='bold')
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Positive Rate')

plt.suptitle('Treatment Recommendations by Evidence Tier', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('_temp_tier_recs.png', dpi=150, bbox_inches='tight')
plt.show()

display(HTML("""<p><b>Reading the tiers:</b> Strong tier requires both large sample size (n≥30) and statistical significance — no PSSD treatment reaches this bar yet. Moderate tier includes bupropion (most data but middling results). Preliminary tier contains the most interesting signals (antihistamines, microdosing, ketogenic diet) but with sample sizes too small for confident conclusions. The dashed line at 50% marks the chance baseline.</p>"""))


## Conclusion

Based on 902 treatment reports from 220 users in r/PSSD, the data tells a clear story even if the sample sizes demand humility about the conclusions.

The most important finding is what is *not* working: bupropion, the community's default alternative to SSRIs, helps roughly half the people who try it — not significantly better than chance. Its prominence in community discussions reflects prescriber habits and its status as "the non-SSRI antidepressant," not demonstrated efficacy for PSSD recovery. Patients and clinicians should calibrate expectations accordingly. Bupropion may still be worth trying (48% positive is not zero), but it should not be treated as the primary PSSD treatment.

The most surprising finding is the antihistamine signal. Loratadine, cetirizine, ketotifen, and the antihistamine class collectively show the highest positive rates in this dataset. These are cheap, widely available, low-risk drugs that no clinical guideline recommends for sexual dysfunction. Their consistent positive signal across multiple compounds within the same mechanism class is exactly the kind of pattern that warrants formal investigation. The most parsimonious explanation is neuroinflammation: if PSSD involves mast cell activation or histamine dysregulation in the CNS or peripheral nervous system, antihistamines would directly address that component. This aligns with emerging case reports of anti-inflammatory approaches to PSSD.

A patient asking "what should I try for PSSD?" should consider the following hierarchy based on this data: (1) antihistamines are the lowest-risk, highest-signal option — loratadine and cetirizine are OTC and well-tolerated; (2) dopaminergic agents like cabergoline or pramipexole show moderate promise but require prescriptions and monitoring; (3) PDE5 inhibitors (tadalafil) address the sexual dysfunction symptom directly with good signal but do not treat the underlying cause; (4) psychedelics and microdosing show interesting preliminary data but carry legal and safety considerations.

The absence of any treatment reaching the Strong evidence tier is itself a finding. PSSD remains a condition where the best available evidence comes from community self-reports, not clinical trials. Every recommendation here is preliminary. But the antihistamine signal is strong enough and the intervention is safe enough that it deserves immediate clinical attention.

## Research Limitations

**1. Selection bias:** Reddit users are not representative of all PSSD patients. They skew younger, more tech-savvy, and potentially more severe (people with mild or resolved PSSD may leave the community and never post).

**2. Reporting bias:** Users are more likely to report dramatic experiences — either breakthroughs or failures. Treatments that produce modest, gradual improvement may be systematically underrepresented because they do not make compelling posts.

**3. Survivorship bias:** We can only analyze people who are still posting. Patients who fully recovered may have left the community. Patients whose condition worsened catastrophically may also have stopped posting. The "recovery-mentioning" cohort likely overrepresents partial recovery and underrepresents full recovery.

**4. Recall bias:** Users describe treatment experiences from memory, often weeks or months after the fact. The timing, dosage, and concurrent treatments may be inaccurately recalled. Retrospective self-assessment of sexual function is particularly unreliable.

**5. Confounding:** Most users try multiple treatments simultaneously or sequentially. When a user reports improvement while taking antihistamines, they may also be taking supplements, changing diet, or experiencing natural fluctuation. We cannot isolate individual treatment effects without a controlled design.

**6. No control group:** There is no untreated comparison group. Some PSSD patients improve spontaneously over time. Without knowing the natural recovery rate, we cannot distinguish treatment effects from natural remission. Every positive rate in this analysis could be inflated by spontaneous recovery.

**7. Sentiment vs efficacy:** Our outcome measure is user-reported sentiment, not clinical measurement of sexual function. A user who says "bupropion helped my mood but not my PSSD" might generate a "mixed" sentiment score even though the treatment failed for the target condition. Sentiment is a noisy proxy for treatment efficacy.

**8. Temporal snapshot:** This data covers one month (2026-03-12 to 2026-04-11). Treatment trends, community norms, and discussion patterns may differ at other time points. A treatment that was trending during this period may appear overrepresented.

In [ ]:
display(HTML('''<div style="margin-top: 30px; padding: 20px; background-color: #fff3cd; border: 1px solid #ffc107; border-radius: 5px;">
<p style="font-size: 1.2em; font-weight: bold; font-style: italic;">These findings reflect reporting patterns in online communities, not population-level treatment effects. This is not medical advice.</p>
</div>'''))
